<a href="https://colab.research.google.com/github/kaushikpatriot/ML-Projects/blob/main/DecisionTrees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
eps = np.finfo(float).eps
eps

2.220446049250313e-16

In [ ]:
dfm = pd.read_csv('Loandata.csv')

In [ ]:
dfm

,Age,Married,Salary,Home_owner,Loan_worthy
0,junior,yes,high,yes,yes
1,middle,no,low,yes,no
2,senior,no,low,no,no
3,senior,no,low,yes,no
4,middle,yes,high,yes,yes
5,junior,no,high,yes,yes
6,junior,yes,low,yes,yes
7,middle,yes,high,no,yes
8,middle,no,low,no,no
9,junior,no,low,no,no


In [ ]:
def entire_entropy(df):
  target = df.keys()[-1]
  classes = df[target].unique()
  entropy = 0
  n_samples = len(df)
  for i in classes:
    i_count = df[target].value_counts()[i]
    p_i = i_count/n_samples
    entropy += -1*p_i*np.log2(p_i+eps)
  return entropy

entire_entropy(dfm)

0.9774178175281709

In [ ]:
def entire_gini(df):
  target = df.keys()[-1]
  classes = df[target].unique()
  gini = 0
  n_samples = len(df)
  for i in classes:
    i_count = df[target].value_counts()[i]
    p_i = i_count/n_samples
    gini += p_i*(1-p_i)
  return gini

entire_gini(dfm)

0.4844290657439446

In [ ]:
def entropy_attribute(df,attribute):
  target = df.keys()[-1]
  labels = df[target].unique()
  att_un = df[attribute].unique()
  n_samples = len(df)
  entropy=0
  for attval in att_un:
    node_entropy = 0
    #proportion of dataset for the given attribute in the total dataset
    cnt_att = len(df[attribute][df[attribute]==attval])
    prop = cnt_att / n_samples
    for label in labels:
      #proportion of dataset for the given attribute and given label over the subset of records for the attribute
      cnt_att_label = len(df[attribute][df[attribute]==attval][df[target]==label])
      p_i = cnt_att_label / (cnt_att+eps)
      node_entropy += -p_i * np.log2(p_i+eps)
    entropy += prop*node_entropy
  return entropy

def best_attribute_to_divide(df):
  feats = df.keys()[:-1]
  entropy_list = []
  for i in feats:
    entropy_list.append(entropy_attribute(df,i))
  return df.keys()[np.argmin(entropy_list)]

best_attribute_to_divide(dfm)


'Married'

###Regression

In [ ]:
present_price = np.array([5.59,9.54,9.85,4.15,6.87,9.83,8.12,8.61,8.89,8.92]).reshape(10,-1)
km_driven = np.array([27000,43000,6900,5200,42450,2071,18796,33429,20273,42376]).reshape(10,-1)
age = np.array([8,9,5,11,8,4,7,7,6,7]).reshape(10,-1)
selling_price = np.array([3.35,4.75,7.25,2.85,4.6,9.25,6.75,6.5,8.75,7.45]).reshape(10,-1)

In [ ]:
X = np.concatenate([present_price,km_driven,age,selling_price],axis=1)
X.shape

(10, 4)

In [ ]:
def split(dataset, featureindex, threshold):
  #left_child = np.array([row for row in dataset if row[featureindex] <= threshold])
  #right_child = np.array([row for row in dataset if row[featureindex] > threshold])
  left_child = X[np.where(X[:, featureindex] <= threshold)]
  right_child = X[np.where(X[:,featureindex] > threshold)]
  return left_child, right_child

split(X,0,7)

(array([[5.590e+00, 2.700e+04, 8.000e+00, 3.350e+00],
        [4.150e+00, 5.200e+03, 1.100e+01, 2.850e+00],
        [6.870e+00, 4.245e+04, 8.000e+00, 4.600e+00]]),
 array([[9.5400e+00, 4.3000e+04, 9.0000e+00, 4.7500e+00],
        [9.8500e+00, 6.9000e+03, 5.0000e+00, 7.2500e+00],
        [9.8300e+00, 2.0710e+03, 4.0000e+00, 9.2500e+00],
        [8.1200e+00, 1.8796e+04, 7.0000e+00, 6.7500e+00],
        [8.6100e+00, 3.3429e+04, 7.0000e+00, 6.5000e+00],
        [8.8900e+00, 2.0273e+04, 6.0000e+00, 8.7500e+00],
        [8.9200e+00, 4.2376e+04, 7.0000e+00, 7.4500e+00]]))

In [ ]:
def variance_reduction(parent, lchild, rchild):
  parent_var = np.var(parent)
  lchild_var = np.var(lchild)
  rchild_var = np.var(rchild)
  childvar = (len(lchild)/len(parent)) * lchild_var + (len(rchild)/len(parent)) * rchild_var
  return parent_var - childvar

lchild, rchild = split(X,0,  7)
variance_reduction(X[:,-1], lchild[:,-1], rchild[:,-1])


2.786785714285714

In [ ]:
def node_value(Y):
  val = np.mean(Y)
  return val

node_value(rchild[:,-1])

7.242857142857143

In [ ]:
def get_best_split(dataset,num_samples,num_features):
  best_split = {}
  max_reduction = -float('inf')
  for index in range(num_features):
    possible_values = np.unique(dataset[:, index])
    for threshold in possible_values:
      left_data, right_data = split(dataset, index, threshold)
      if (len(left_data) > 0 and len(right_data) > 0):
        cur_var_red = variance_reduction(dataset[:,-1],left_data[:,-1], right_data[:,-1])

        if cur_var_red > max_reduction:
          best_split['feature_index'] = index
          best_split['threshold'] = threshold
          best_split['dataset_left'] = left_data
          best_split['dataset_right'] = right_data
          best_split['max_var_reduction'] = cur_var_red

  return best_split

n_samples, n_features = 10, 3
get_best_split(X,n_samples,n_features)



{'dataset_left': array([[5.5900e+00, 2.7000e+04, 8.0000e+00, 3.3500e+00],
        [9.5400e+00, 4.3000e+04, 9.0000e+00, 4.7500e+00],
        [9.8500e+00, 6.9000e+03, 5.0000e+00, 7.2500e+00],
        [6.8700e+00, 4.2450e+04, 8.0000e+00, 4.6000e+00],
        [9.8300e+00, 2.0710e+03, 4.0000e+00, 9.2500e+00],
        [8.1200e+00, 1.8796e+04, 7.0000e+00, 6.7500e+00],
        [8.6100e+00, 3.3429e+04, 7.0000e+00, 6.5000e+00],
        [8.8900e+00, 2.0273e+04, 6.0000e+00, 8.7500e+00],
        [8.9200e+00, 4.2376e+04, 7.0000e+00, 7.4500e+00]]),
 'dataset_right': array([[4.15e+00, 5.20e+03, 1.10e+01, 2.85e+00]]),
 'feature_index': 2,
 'max_var_reduction': 1.209999999999999,
 'threshold': 9.0}

###Decision Tree for Classification

In [ ]:
def buildTree(df, tree=None):
  target = df.keys()[-1]
  node = best_attribute_to_divide(df)
  attValue = np.unique(df[node])
  if tree == None:
    tree ={}
    tree[node] ={}

  for value in attValue:
    subtable = df[df[node]==value].reset_index(drop=True)
    clValue,counts = np.unique(subtable['play'],return_counts = True)

    if len(counts) == 1:
      tree[node][value] = clValue[0]
    else:
      tree[node][value] = buildTree(subtable)

  return tree
